# Does translation style, not the tokenizer, drive the cost gap?

**The question.** Could differences between languages reflect how each language was translated,
rather than the tokenizer? The translator study shows each language's file is representative of that
language. It does not show that different languages carry the same meaning in the same amount of
text. If some translation traditions add a lot of explanatory material, those languages look expensive
under every tokenizer, and part of what the paper calls tokenizer inequality would really be
translation style.

This notebook runs two automatic checks. Neither needs human raters.

| Part | Check | Needs | Time |
|---|---|---|---|
| **A** | Remove bracketed explanatory insertions and footnote markers, recompute fertility under every tokenizer, and test whether the cost rankings, the p90/p10 inequality and the script-fit result survive | CPU | ~15 min |
| **B** | Score every translation against the **Arabic source** with a reference-free quality-estimation model (CometKiwi), then test whether adequacy tracks cost, and whether rankings hold on the verses rendered adequately in every language | GPU (T4) + a free Hugging Face token | ~25 min at the default 2,000-verse sample |

Part A is the direct test and stands on its own. Part B is optional
supporting evidence. **Run Part A first**; Part B reuses its per-verse token counts.

**Upload** when prompted (all three are in this repository (`data/` and `results/`)):

- `translations.zip` (required)
- `joint_language_table_corrected.csv` (required: script and Joshi tier)
- `results/superseded/fertility_production_v2_all11.csv` (optional: the August counts, to check
  reproduction; eight tokenizers match exactly, mT5 and XLM-R do not, see the README)

**Conventions match the paper.** Fertility statistics run over the 41 translation files; correlations
against per-language covariates use $n=40$ (one Bosnian translation, `bosnian-rwwad`); Aya-101 is
omitted because it shares mT5's vocabulary; ByT5 is kept for tokens per verse but dropped from every
tokens-per-byte test, since its tokens per byte is 1 by construction.

## 0. Setup

In [ ]:
!pip -q install tiktoken sentencepiece tokenizers protobuf
import os, io, re, csv, json, glob, zipfile, time, subprocess, sys
import numpy as np, pandas as pd
from scipy import stats

SEED         = 42
OUT          = "results_adequacy"
BOSNIAN_DROP = "bosnian-mihanovich"      # one Bosnian for n=40 analyses, as in the paper
N_PERM       = 20000                     # Freedman-Lane permutations, as in the paper
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("ready")

In [ ]:
# [colab-only] upload inputs
try:
    from google.colab import files
    need = [f for f in ["translations.zip", "joint_language_table_corrected.csv"] if not os.path.exists(f)]
    if need:
        print("Upload:", need, "(and optionally fertility_production_v2_all11.csv)")
        files.upload()
except ImportError:
    pass
for f in ["translations.zip", "joint_language_table_corrected.csv"]:
    assert os.path.exists(f), f"missing {f}"
print("inputs present; published fertility file:", os.path.exists("fertility_production_v2_all11.csv"))

## 1. Load the corpus

The reader and the language naming are copied from `01_tokenizer_fertility.ipynb`, so file names,
verse keys and text are exactly what the published fertility numbers were computed on.

In [ ]:
if not os.path.exists("corpus"):
    with zipfile.ZipFile("translations.zip") as z:
        z.extractall("corpus")
csv_files = sorted(glob.glob("corpus/**/*.csv", recursive=True))

def load_csv(fp):
    keys = {}
    with open(fp, encoding="utf-8", errors="replace") as f:
        rdr = csv.reader(f); started = False; iS = iA = iT = None
        for row in rdr:
            if not started:
                low = [(c or "").strip().lower() for c in row]
                if "sura" in low and "aya" in low:
                    started = True; iS = low.index("sura"); iA = low.index("aya")
                    iT = low.index("translation") if "translation" in low else 3
                continue
            try:
                keys[(int(row[iS]), int(row[iA]))] = row[iT]
            except Exception:
                continue
    return keys

lang_name = lambda fp: os.path.basename(fp).split("_")[0]
from collections import Counter
cnt  = Counter(lang_name(f) for f in csv_files)
disp = {f: (lang_name(f) if cnt[lang_name(f)] == 1
            else lang_name(f) + "-" + os.path.basename(f).split("_")[1]) for f in csv_files}
TEXT = {disp[f]: load_csv(f) for f in csv_files}
NAMES = sorted(TEXT)                                   # 41 translation files
KEYS = sorted(set.intersection(*[set(v) for v in TEXT.values()]))
print(f"{len(NAMES)} translation files, {len(KEYS)} common verses")
assert len(NAMES) == 41 and len(KEYS) == 6236

meta = pd.read_csv("joint_language_table_corrected.csv")[["language", "script", "joshi"]]
meta["script5"] = meta.script.replace({"Khmer": "Brahmic"})   # the paper's five-group split
meta = meta.set_index("language")
missing = set(NAMES) - set(meta.index)
assert not missing, f"no metadata for {missing}"
N40 = [n for n in NAMES if n != BOSNIAN_DROP]
print("n =", len(N40), "for covariate analyses")

## A1. Find explanatory insertions

Quran translations commonly mark words the translator added for clarity, and they mark them in
different ways. What counts here:

| Removed | Pattern | Why |
|---|---|---|
| footnote markers | `[12]` | pointers to the separate footnotes column, not translation text |
| verse numbers | `113. …` at the start (French, Lithuanian, Macedonian), `(43)` (Uyghur) | the file repeats the aya number inside the text |
| glossary asterisks | `アッラー*` (Japanese) | pointers to a glossary, not translation text |
| parenthetical insertions | `( … )` | the most common convention for added explanation |
| full-width parentheses | `（ … ）` | the same convention in Japanese |
| square-bracket insertions | `[ … ]` (non-numeric) | used by the Persian, Tajik, Romanian and Tagalog files, among others |

The first three are corpus artifacts rather than translation choices; they are removed in a
`clean` condition of their own, so their effect on the published numbers is reported separately from
the effect of the insertions. Nested brackets are removed innermost first until none remain.

**Not removed:** guillemets `« … »`. The French, Spanish, Uzbek, Persian and Uyghur files use them as
quotation marks for reported speech, which is part of the verse, not added commentary.

**What this cannot catch.** It only detects explanation the translator *marked*. Unmarked added
material is invisible to it, which is the gap Part B is meant to cover.

**Check the examples printed below before trusting the numbers.** Skim a few per language and confirm
the highlighted spans ⟦like this⟧ really are added explanation. If a language uses brackets for
something else, add it to `EXCLUDE_LANGS` and re-run.

In [ ]:
EXCLUDE_LANGS = set()     # languages whose brackets turn out not to be insertions

FOOTNOTE  = re.compile(r"\[\d+\]")
INSERT    = [re.compile(r"\([^()]*\)"), re.compile(r"（[^（）]*）"), re.compile(r"\[[^\[\]]*\]")]
PUNCT_FIX = re.compile(r"\s+([,.;:!?،؛。、」』)])")

def clean_markers(t, aya):
    # corpus artifacts: footnote markers, repeated verse numbers, glossary asterisks
    t = FOOTNOTE.sub("", t)
    t = re.sub(r"\(\s*%d\s*\)" % aya, "", t)            # Uyghur: (43)
    t = re.sub(r"^\s*%d\s*[.)\-–]?\s+" % aya, "", t)     # French, Lithuanian, Macedonian: "113. "
    return t.replace("*", "")                             # Japanese glossary markers

def remove_spans(t, sep=""):
    prev = None
    while prev != t:
        prev = t
        for p in INSERT:
            t = p.sub(sep, t)
    return t

def tidy(t):
    t = PUNCT_FIX.sub(r"\1", t)
    return re.sub(r"\s{2,}", " ", t).strip()

def strip_insertions(t, lang, aya):
    c = clean_markers(t, aya)
    return c if lang in EXCLUDE_LANGS else tidy(remove_spans(c, " "))

def highlight(t):
    for p in INSERT:
        t = p.sub(lambda m: "⟦" + m.group(0) + "⟧", t)
    return t

nb = lambda s: len(s.encode("utf-8"))
COND = {"raw":      lambda t, l, a: t,
        "clean":    lambda t, l, a: clean_markers(t, a),
        "stripped": strip_insertions}
VARIANT = {c: {n: [f(str(TEXT[n][k]), n, k[1]) for k in KEYS] for n in NAMES} for c, f in COND.items()}
BYTES = {c: np.array([[nb(t) for t in VARIANT[c][n]] for n in NAMES]) for c in COND}
HAS = {n: np.array([n not in EXCLUDE_LANGS and any(p.search(t) for p in INSERT)
                    for t in VARIANT["clean"][n]]) for n in NAMES}

rng = np.random.default_rng(SEED)
rows = []
for i, n in enumerate(NAMES):
    span_bytes = sum(nb(t) - nb(remove_spans(t)) for t, h in zip(VARIANT["clean"][n], HAS[n]) if h)
    rows.append({"language": n, "script": meta.loc[n, "script"], "joshi": int(meta.loc[n, "joshi"]),
                 "verses_with_insertion": round(HAS[n].mean(), 3),
                 "insertion_byte_share": round(span_bytes / BYTES["clean"][i].sum(), 4),
                 "artifact_byte_share": round(1 - BYTES["clean"][i].sum() / BYTES["raw"][i].sum(), 4),
                 "verses_empty_after_strip": int(sum(1 for t in VARIANT["stripped"][n] if not t))})
INS = pd.DataFrame(rows).set_index("language").sort_values("insertion_byte_share", ascending=False)
INS.to_csv(f"{OUT}/insertion_rates.csv")
print(INS.to_string())
print(f"\nmedian insertion byte share {INS.insertion_byte_share.median():.3f}, "
      f"max {INS.insertion_byte_share.max():.3f} ({INS.insertion_byte_share.idxmax()})")

In [ ]:
# Eyeball check: two random verses with insertions per language (first 220 characters)
for n in INS.index:
    idx = np.flatnonzero(HAS[n])
    if not len(idx):
        continue
    print(f"\n=== {n}  ({INS.loc[n, 'insertion_byte_share']:.1%} of bytes)")
    for j in rng.choice(idx, size=min(2, len(idx)), replace=False):
        print("  ", KEYS[j], highlight(VARIANT["clean"][n][j])[:220].replace("\n", " "))

## A2. Load the tokenizers

Same specs and the same fallback mirrors as `01_tokenizer_fertility.ipynb`, so each tokenizer resolves
to the same vocabulary the paper used. Aya-101 is skipped because it ships mT5's vocabulary.

In [ ]:
# [tokenizers]
import tiktoken
from transformers import AutoTokenizer

def make_tiktoken(name):
    enc = tiktoken.get_encoding(name)
    return lambda texts: [len(t) for t in enc.encode_batch(texts)]

def make_hf(model_id):
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
    except Exception:
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    return lambda texts: [len(ids) for ids in tok(texts, add_special_tokens=False)["input_ids"]]

specs = [
    ("gpt4o_o200k", "tiktoken", "o200k_base", []),
    ("gpt4_cl100k", "tiktoken", "cl100k_base", []),
    ("mbert",  "hf", "bert-base-multilingual-cased", []),
    ("xlmr",   "hf", "xlm-roberta-base", []),
    ("bloom",  "hf", "bigscience/bloom-560m", []),
    ("byt5",   "hf", "google/byt5-small", []),
    ("nllb",   "hf", "facebook/nllb-200-distilled-600M", []),
    ("mt5",    "hf", "google/mt5-small", []),
    ("llama3", "hf", "meta-llama/Meta-Llama-3-8B", ["NousResearch/Meta-Llama-3-8B", "unsloth/llama-3-8b"]),
    ("gemma",  "hf", "google/gemma-7b", ["unsloth/gemma-2-2b", "unsloth/gemma-2b"]),
]
TOKENIZERS = {}
for name, kind, primary, fallbacks in specs:
    for ref in [primary] + fallbacks:
        try:
            TOKENIZERS[name] = make_tiktoken(ref) if kind == "tiktoken" else make_hf(ref)
            print("loaded", name, "<-", ref); break
        except Exception:
            continue
    else:
        print("SKIPPED", name)
print("\nactive:", list(TOKENIZERS))

In [ ]:
LABEL = {"nllb": "NLLB", "mt5": "mT5 / Aya-101", "gpt4o_o200k": "GPT-4o (o200k)", "mbert": "mBERT",
         "xlmr": "XLM-R", "gemma": "Gemma-2", "bloom": "BLOOM", "byt5": "ByT5 (bytes)",
         "gpt4_cl100k": "GPT-4 (cl100k)", "llama3": "Llama-3"}
TOKS    = [t for t in LABEL if t in TOKENIZERS]
LEARNED = [t for t in TOKS if t != "byt5"]        # tokens/byte is constant for ByT5

## A3. Tokenize every file three ways

`raw` is the text exactly as the paper measured it; `clean` drops the artifacts only (footnote
markers, repeated verse numbers, asterisks); `stripped` also drops the insertions. Per-verse counts are saved so Part B can reuse them.

The first check is **reproduction**: `raw` must match the published fertility. If it does not, stop,
because nothing after it would be comparable.

In [ ]:
t0 = time.time()
COUNTS = {c: {} for c in COND}
for t in TOKS:
    for c in COND:
        COUNTS[c][t] = np.array([TOKENIZERS[t](VARIANT[c][n]) for n in NAMES], dtype=np.int32)
    print(f"  {t:12s} done  ({time.time()-t0:.0f}s)", flush=True)
np.savez_compressed(f"{OUT}/per_verse_tokens.npz", names=np.array(NAMES),
                    **{f"{c}__{t}": COUNTS[c][t] for c in COND for t in TOKS},
                    **{f"bytes__{c}": BYTES[c] for c in COND})

TPV = {c: pd.DataFrame({t: COUNTS[c][t].mean(1) for t in TOKS}, index=NAMES) for c in COND}
TPB = {c: pd.DataFrame({t: COUNTS[c][t].sum(1) / BYTES[c].sum(1) for t in TOKS}, index=NAMES) for c in COND}

if os.path.exists("fertility_production_v2_all11.csv"):
    pub = pd.read_csv("fertility_production_v2_all11.csv").set_index("language")
    rep = {t: float(np.max(np.abs(TPV["raw"][t] / pub.loc[NAMES, f"{t}_tok_per_verse"] - 1)))
           for t in TOKS if f"{t}_tok_per_verse" in pub}
    byt = float(np.max(np.abs(BYTES["raw"].mean(1) / pub.loc[NAMES, "bytes_per_verse"] - 1)))
    print("\nReproduction, max relative difference from the published fertility:")
    print(f"  bytes/verse {byt:.2e}")
    for t, v in rep.items():
        print(f"  {t:12s} {v:.2e}", "" if v < 1e-3 else "  <-- DOES NOT REPRODUCE: check the tokenizer version")
else:
    print("published file not uploaded; reproduction check skipped")

## A4. Do the cost rankings and the inequality survive?

For each tokenizer, over the 41 translation files:

- **p90/p10** on tokens per verse, raw and stripped (the paper's Table 1 measure)
- **ρ(raw, stripped)**: rank agreement of the 41 files before and after removing insertions, on
  tokens per verse and on tokens per byte
- **largest rank move**: which file moves most and how far

Then, over the 40 languages: does a language's insertion rate predict its cost? If translation style
were driving the gap, languages with more inserted material would sit higher on tokens per verse.

In [ ]:
p9010 = lambda x: np.percentile(x, 90) / np.percentile(x, 10)
rows = []
for t in TOKS:
    a, b = TPV["raw"][t], TPV["stripped"][t]
    ra, rb = a.rank(ascending=False), b.rank(ascending=False)
    mv = (ra - rb).abs()
    r = {"tokenizer": LABEL[t],
         "p90p10_raw": round(p9010(a), 3), "p90p10_clean": round(p9010(TPV["clean"][t]), 3),
         "p90p10_stripped": round(p9010(b), 3),
         "p90p10_change_pct": round(100 * (p9010(b) / p9010(a) - 1), 1),
         "rho_tpv_raw_vs_stripped": round(stats.spearmanr(a, b)[0], 4),
         "rho_tpb_raw_vs_stripped": (round(stats.spearmanr(TPB["raw"][t], TPB["stripped"][t])[0], 4)
                                      if t in LEARNED else np.nan),
         "largest_rank_move": int(mv.max()), "moved_most": mv.idxmax()}
    rows.append(r)
ROBUST = pd.DataFrame(rows)
ROBUST.to_csv(f"{OUT}/robustness_to_insertions.csv", index=False)
print(ROBUST.to_string(index=False))

In [ ]:
ins = INS.loc[N40, "insertion_byte_share"]
rows = []
for t in TOKS:
    r1, p1 = stats.spearmanr(ins, TPV["raw"].loc[N40, t])
    row = {"tokenizer": LABEL[t], "rho_ins_vs_tpv": round(r1, 3), "p_tpv": round(p1, 4)}
    if t in LEARNED:
        r2, p2 = stats.spearmanr(ins, TPB["raw"].loc[N40, t])
        row.update({"rho_ins_vs_tpb": round(r2, 3), "p_tpb": round(p2, 4)})
    rows.append(row)
INSCORR = pd.DataFrame(rows)
INSCORR.to_csv(f"{OUT}/insertion_vs_cost.csv", index=False)
print("Does a language's insertion rate predict its cost?  (Spearman, n = 40)")
print(INSCORR.to_string(index=False))
print("\nat n = 40, |rho| below about 0.32 is indistinguishable from zero (the paper's bound)")

# which languages get cheaper when insertions go, under GPT-4o
t = "gpt4o_o200k" if "gpt4o_o200k" in TOKS else TOKS[0]
drop = (1 - TPV["stripped"][t] / TPV["raw"][t]).sort_values(ascending=False)
print(f"\nLargest cost reductions under {LABEL[t]} after stripping:")
print((100 * drop.head(10)).round(1).to_string())

## A5. Does the script-fit result survive?

This reruns the paper's Section 6.4 test on the stripped text: variance in log tokens per byte uniquely
explained by the five script groups after controlling Joshi tier, Freedman–Lane permutation,
20,000 permutations, Holm-corrected across the nine learned vocabularies. The `raw` column should
reproduce the paper's table; the `stripped` column is the new number.

In [ ]:
RNG = np.random.default_rng(SEED)

def ols_r2(y, X):
    X = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    fit = X @ beta; res = y - fit
    return 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum(), res, fit

def dummies(labels):
    cats = sorted(set(labels))[1:]
    return np.column_stack([(np.asarray(labels) == c).astype(float) for c in cats])

def freedman_lane(y, script_labels, tier, n_perm=N_PERM, rng=RNG):
    tier = np.asarray(tier, float).reshape(-1, 1)
    full = np.column_stack([tier, dummies(script_labels)])
    r2_red, res_red, fit_red = ols_r2(y, tier)
    r2_full = ols_r2(y, full)[0]
    d_obs = r2_full - r2_red
    null = np.empty(n_perm)
    for i in range(n_perm):
        ys = fit_red + res_red[rng.permutation(len(y))]
        null[i] = ols_r2(ys, full)[0] - ols_r2(ys, tier)[0]
    return d_obs, (1 + (null >= d_obs).sum()) / (1 + n_perm)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o):
        run = max(run, (m - k) * p[i]); adj[i] = min(run, 1.0)
    return adj

sc, tier = meta.loc[N40, "script5"].values, meta.loc[N40, "joshi"].values
rows = []
for t in LEARNED:
    r = {"tokenizer": LABEL[t]}
    for c in ["raw", "stripped"]:
        d, p = freedman_lane(np.log(TPB[c].loc[N40, t].values), sc, tier)
        r[f"dR2_{c}"], r[f"p_{c}"] = round(d, 3), p
    rows.append(r)
    print(f"  {LABEL[t]:16s} dR2 raw {r['dR2_raw']:.3f}   stripped {r['dR2_stripped']:.3f}", flush=True)
SCRIPT = pd.DataFrame(rows)
for c in ["raw", "stripped"]:
    SCRIPT[f"p_holm_{c}"] = holm(SCRIPT[f"p_{c}"]).round(4)
SCRIPT.to_csv(f"{OUT}/script_fit_after_stripping.csv", index=False)
print()
print(SCRIPT.to_string(index=False))
print(f"\nsignificant after Holm: raw {(SCRIPT.p_holm_raw < .05).sum()}/{len(SCRIPT)}, "
      f"stripped {(SCRIPT.p_holm_stripped < .05).sum()}/{len(SCRIPT)}")

## A6. English-anchored ratios

The paper's cost ratios relative to English (for example Khmer to English under GPT-4o, $3.75\times$)
are the numbers most exposed to translation style, because they depend on two individual files rather
than on the whole distribution.

In [ ]:
PAIRS = [("khmer", "english"), ("malayalam", "english"), ("tamil", "english"), ("japanese", "english")]
rows = []
for t in ["gpt4o_o200k", "gpt4_cl100k", "llama3"]:
    if t not in TOKS:
        continue
    for a, b in PAIRS:
        rows.append({"tokenizer": LABEL[t], "pair": f"{a}/{b}",
                     "ratio_raw": round(TPV["raw"].loc[a, t] / TPV["raw"].loc[b, t], 2),
                     "ratio_stripped": round(TPV["stripped"].loc[a, t] / TPV["stripped"].loc[b, t], 2)})
RATIOS = pd.DataFrame(rows)
RATIOS.to_csv(f"{OUT}/english_anchored_ratios.csv", index=False)
print(RATIOS.to_string(index=False))

# Part B — reference-free adequacy against the Arabic source

**What it measures.** CometKiwi (`Unbabel/wmt22-cometkiwi-da`) reads a source sentence and a
translation and predicts a human direct-assessment score, with no reference translation. Here the
source is the Arabic verse and the "translation" is each language's rendering.

**What it cannot do, stated up front.**

- CometKiwi was trained mostly on English-centred pairs; Arabic-source directions are zero-shot.
  Scores are therefore more trustworthy *within* a language (which verses are rendered worse) than
  *between* languages. The subset analysis below uses within-language z-scores for that reason.
- It is built on InfoXLM, which covers XLM-R's 100 languages. **Yoruba, Lingala, Mooré, Asante Twi and
  Tajik are outside that set**; they are scored but flagged and excluded from every adequacy analysis.
- QE models are known to be weakly sensitive to *added* material. The diagnostic in B3 measures this
  on our data. If the model does not penalise marked insertions, Part B speaks to omission and
  mistranslation, and Part A remains the test for verbosity.

**Access.** The model is gated. Once: log in at huggingface.co, open
`huggingface.co/Unbabel/wmt22-cometkiwi-da`, accept the terms, then create a read token under
Settings → Access Tokens. In Colab, add it under the key icon (Secrets) as `HF_TOKEN` with notebook
access switched on.

**Isolation.** `unbabel-comet` pins `numpy<2` and older `transformers`, which conflict with current
Colab. Instead of downgrading your session, the scorer runs in a separate Python environment
created with `uv`, so nothing in Part A is disturbed and no runtime restart is needed.

In [ ]:
SAMPLE_N  = 2000        # verses scored per language; None = all 6,236 (about 3x longer)
QE_BATCH  = 64
QE_UNSUPPORTED = {"yoruba", "lingala", "moore", "asante", "tajik"}

import torch
GPU = torch.cuda.is_available()
print("GPU:", GPU)
if not GPU:
    print("WARNING: Part B on CPU would take many hours. Runtime -> Change runtime type -> T4.")

In [ ]:
# [colab-only] isolated scorer environment (first run about 3 minutes)
# The scorer environment gets the same torch build as this session (so it matches Colab's GPU
# driver), plus unbabel-comet and the setuptools its old torchmetrics pin still imports from.
QE_PY = "/content/qeenv/bin/python"
if not os.path.exists(QE_PY):
    ver, _, cu = torch.__version__.partition("+")
    index = f"https://download.pytorch.org/whl/{cu or 'cpu'}"
    subprocess.run(f"pip -q install uv && uv venv /content/qeenv -q && "
                   f"uv pip install -q --python {QE_PY} torch=={ver} --index-url {index} && "
                   f"uv pip install -q --python {QE_PY} unbabel-comet 'setuptools<81'",
                   shell=True, check=True)
subprocess.run([QE_PY, "-W", "ignore", "-c",
                "import comet, torch; print('comet', comet.__version__, '| torch', torch.__version__, "
                "'| GPU visible to scorer:', torch.cuda.is_available())"], check=True)

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) after accepting the model's terms."

## B1. The Arabic source

The same Uthmani text the paper's Arabic row came from (`risan/quran-json`). If the download is
blocked, upload `quran.json` from that repository when prompted.

In [ ]:
# [arabic]
import urllib.request
ARABIC_URL = "https://raw.githubusercontent.com/risan/quran-json/main/dist/quran.json"
if not os.path.exists("quran.json"):
    try:
        urllib.request.urlretrieve(ARABIC_URL, "quran.json")
    except Exception as e:
        print("download failed:", e, "-- upload quran.json")
        from google.colab import files; files.upload()
ARABIC = {(s["id"], v["id"]): v["text"] for s in json.load(open("quran.json", encoding="utf-8"))
          for v in s["verses"]}
assert set(KEYS) <= set(ARABIC), "Arabic keys do not cover the corpus"
print("Arabic verses:", len(ARABIC))

## B2. Score

Scores are written per language to `results_adequacy/qe/`, so if Colab disconnects, re-run this cell
and it resumes where it stopped. The text scored is each file with footnote markers removed and
insertions *kept*, because the question is how adequate the text the paper measured is.

In [ ]:
QE_DIR = f"{OUT}/qe"; os.makedirs(QE_DIR, exist_ok=True)
rng_q = np.random.default_rng(SEED)
QIDX = (np.arange(len(KEYS)) if SAMPLE_N is None
        else np.sort(rng_q.choice(len(KEYS), size=SAMPLE_N, replace=False)))
print(f"scoring {len(QIDX)} verses x {len(NAMES)} files")

SCORER = r'''
import sys, json, logging, warnings
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
from comet import download_model, load_from_checkpoint
model = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
jobs = json.load(open(sys.argv[1], encoding="utf-8"))
gpus, bs = int(sys.argv[2]), int(sys.argv[3])
for job in jobs:
    data = [{"src": s, "mt": m} for s, m in zip(job["src"], job["mt"])]
    out = model.predict(data, batch_size=bs, gpus=gpus, progress_bar=False)
    with open(job["out"], "w") as f:
        json.dump([float(x) for x in out.scores], f)
    print("scored", job["name"], flush=True)
'''
open("qe_scorer.py", "w").write(SCORER)

src = [ARABIC[KEYS[i]] for i in QIDX]
jobs = []
for n in NAMES:
    fp = f"{QE_DIR}/{n}.json"
    if os.path.exists(fp):
        continue
    mt = [VARIANT["clean"][n][i] or "." for i in QIDX]
    jobs.append({"name": n, "src": src, "mt": mt, "out": fp})
print(f"{len(jobs)} files to score ({len(NAMES) - len(jobs)} already done)")
if jobs:
    json.dump(jobs, open("qe_jobs.json", "w", encoding="utf-8"), ensure_ascii=False)
    env = dict(os.environ, HF_TOKEN=HF_TOKEN, HUGGING_FACE_HUB_TOKEN=HF_TOKEN, TOKENIZERS_PARALLELISM="false")
    t0 = time.time()
    proc = subprocess.Popen([QE_PY, "qe_scorer.py", "qe_jobs.json", str(int(GPU)), str(QE_BATCH)],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log = []
    for line in proc.stdout:
        log.append(line.rstrip())
        if line.startswith("scored"):
            print(f"[{time.time()-t0:5.0f}s] {line.rstrip()}", flush=True)
    if proc.wait() != 0:
        print("\n".join(log[-40:]))
        raise RuntimeError("scorer failed; the last 40 lines of its output are above. "
                           "A 401/403 means the model terms were not accepted or HF_TOKEN is wrong.")

QE = pd.DataFrame({n: json.load(open(f"{QE_DIR}/{n}.json")) for n in NAMES}, index=[KEYS[i] for i in QIDX])
QE.to_csv(f"{OUT}/qe_scores_per_verse.csv")
print("scores:", QE.shape)

## B3. Does adequacy track cost?

Three analyses, all restricted to the 35 languages CometKiwi supports (40 minus the five above):

1. **Diagnostic.** Within each language, do verses with marked insertions score lower than verses
   without? This says whether the metric can see added material at all.
2. **Language level.** Does a language's mean adequacy correlate with its tokens per verse or tokens
   per byte? It is also reported against Joshi tier, since QE quality itself tracks resource level.
3. **Verse level, the direct test.** Keep only verses rendered adequately in *every* supported
   language (within-language z-score; keep the half of verses whose worst language is best), which
   preserves meaning control because every file keeps the same verses. Recompute fertility on that
   subset and compare rankings and p90/p10 with the same sample before filtering.

In [ ]:
SUP = [n for n in N40 if n not in QE_UNSUPPORTED]
print(f"{len(SUP)} supported languages")

# 1. can the metric see insertions?
diag = []
for n in SUP:
    has = HAS[n][QIDX]
    if has.sum() >= 30 and (~has).sum() >= 30:
        s = QE[n].values
        diag.append({"language": n, "n_with": int(has.sum()),
                     "qe_with": s[has].mean(), "qe_without": s[~has].mean(),
                     "diff": s[has].mean() - s[~has].mean(),
                     "p": stats.mannwhitneyu(s[has], s[~has]).pvalue})
DIAG = pd.DataFrame(diag).round(4)
DIAG.to_csv(f"{OUT}/qe_insertion_diagnostic.csv", index=False)
if len(DIAG):
    print(f"Diagnostic over {len(DIAG)} languages with >=30 verses each way:")
    print(f"  median QE difference (with - without insertions): {DIAG['diff'].median():+.4f}")
    print(f"  languages where insertion verses score lower: {(DIAG['diff'] < 0).sum()}/{len(DIAG)}")
    print(f"  ... significantly lower (p < .05): {((DIAG['diff'] < 0) & (DIAG.p < .05)).sum()}")

In [ ]:
# 2. language-level
mq = QE[SUP].mean()
rows = [{"covariate": "Joshi tier", "rho": stats.spearmanr(mq, meta.loc[SUP, "joshi"])[0],
         "p": stats.spearmanr(mq, meta.loc[SUP, "joshi"])[1]}]
for t in TOKS:
    r, p = stats.spearmanr(mq, TPV["raw"].loc[SUP, t])
    rows.append({"covariate": f"{LABEL[t]} tokens/verse", "rho": r, "p": p})
    if t in LEARNED:
        r, p = stats.spearmanr(mq, TPB["raw"].loc[SUP, t])
        rows.append({"covariate": f"{LABEL[t]} tokens/byte", "rho": r, "p": p})
LANGQE = pd.DataFrame(rows).round(4)
LANGQE.to_csv(f"{OUT}/qe_vs_cost_language_level.csv", index=False)
print(f"Mean adequacy vs covariates (Spearman, n = {len(SUP)}):")
print(LANGQE.to_string(index=False))
mq.sort_values().round(4).to_csv(f"{OUT}/qe_mean_by_language.csv", header=["mean_qe"])

In [ ]:
# 3. verse-level: the same verses, rendered adequately everywhere
Z = (QE[SUP] - QE[SUP].mean()) / QE[SUP].std()
worst = Z.min(axis=1).values
keep = worst >= np.median(worst)
print(f"kept {keep.sum()} of {len(keep)} sampled verses (rendered adequately in every supported language)")

rows = []
for t in TOKS:
    c = COUNTS["raw"][t][:, QIDX]
    b = BYTES["raw"][:, QIDX]
    full, sub = c.mean(1), c[:, keep].mean(1)
    r = {"tokenizer": LABEL[t],
         "p90p10_sample": round(p9010(full), 3), "p90p10_adequate": round(p9010(sub), 3),
         "change_pct": round(100 * (p9010(sub) / p9010(full) - 1), 1),
         "rho_tpv": round(stats.spearmanr(full, sub)[0], 4)}
    if t in LEARNED:
        r["rho_tpb"] = round(stats.spearmanr(c.sum(1) / b.sum(1), c[:, keep].sum(1) / b[:, keep].sum(1))[0], 4)
    rows.append(r)
SUBSET = pd.DataFrame(rows)
SUBSET.to_csv(f"{OUT}/fertility_on_adequate_subset.csv", index=False)
print(SUBSET.to_string(index=False))

## Summary and download

`summary.json` collects the headline numbers reported in the paper's robustness section and
Appendix D.

In [ ]:
S = {"insertions": {
        "median_byte_share": float(INS.insertion_byte_share.median()),
        "max_byte_share": float(INS.insertion_byte_share.max()),
        "max_language": INS.insertion_byte_share.idxmax(),
        "n_languages_over_10pct": int((INS.insertion_byte_share > .10).sum())},
     "robustness": ROBUST.set_index("tokenizer")[["p90p10_raw", "p90p10_stripped", "p90p10_change_pct",
                     "rho_tpv_raw_vs_stripped", "rho_tpb_raw_vs_stripped"]].to_dict(orient="index"),
     "insertion_vs_cost": INSCORR.set_index("tokenizer").to_dict(orient="index"),
     "script_fit": {"n_sig_raw": int((SCRIPT.p_holm_raw < .05).sum()),
                    "n_sig_stripped": int((SCRIPT.p_holm_stripped < .05).sum()),
                    "n_vocabularies": len(SCRIPT),
                    "dR2_stripped_range": [float(SCRIPT.dR2_stripped.min()), float(SCRIPT.dR2_stripped.max())]},
     "english_anchored": RATIOS.to_dict(orient="records")}
if "SUBSET" in globals():
    S["adequacy"] = {"n_supported": len(SUP), "sample_n": int(len(QIDX)), "kept": int(keep.sum()),
                     "diag_median_diff": float(DIAG["diff"].median()) if len(DIAG) else None,
                     "diag_lower": f"{int((DIAG['diff'] < 0).sum())}/{len(DIAG)}" if len(DIAG) else None,
                     "language_level": LANGQE.set_index("covariate").to_dict(orient="index"),
                     "subset": SUBSET.set_index("tokenizer").to_dict(orient="index")}
json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str, ensure_ascii=False)
print(json.dumps(S, indent=1, default=str, ensure_ascii=False)[:6000])

import shutil
shutil.make_archive("results_adequacy", "zip", OUT)
try:
    from google.colab import files; files.download("results_adequacy.zip")
except Exception:
    print("zip written: results_adequacy.zip")

## Reading the results

Decide what counts as a problem *before* looking, and report whichever way it comes out.

| Result | What it would mean |
|---|---|
| ρ(raw, stripped) on tokens per verse ≥ 0.95 for every tokenizer and p90/p10 moves by a few percent | Marked insertions do not drive the cost gap. Say so in §6.9 with the numbers. |
| p90/p10 falls substantially for some tokenizer | Part of that tokenizer's reported inequality is translation style. Report the stripped value alongside the raw one in Table 1 and soften the claim for it. |
| ρ(insertion share, tokens per verse) significant and positive | Languages with more inserted material cost more. The raw fertility comparison is partly confounded and the stripped numbers should lead. |
| Script ΔR² on tokens per byte survives Holm for all nine vocabularies after stripping | The paper's central mechanism result does not depend on translation style. |
| An English-anchored ratio moves a lot (for example Khmer/English) | Report both values. This is the same sensitivity already flagged for the English file. |
| B3 diagnostic: insertion verses score no lower | CometKiwi cannot see added material here; Part B supports adequacy (omission, mistranslation), not verbosity. Say that. |
| Rankings on the adequate subset agree with the full sample (ρ ≥ 0.95) | Verses that may be poorly rendered somewhere are not what orders the languages. |

Whatever comes out, the claim stays bounded. These checks can show that *marked* insertions and
*model-detectable* inadequacy do not drive the results. They cannot show semantic equivalence, and the
paper should keep saying so.